# 05. 방법 3: 부분 파인튜닝

이미지 인코더의 마지막 Transformer 블록과 분류층만 학습하고 나머지는 고정한다. 런타임: GPU.

- 분류층 가중치는 제로샷의 국가별 문장 임베딩으로 초기화한다. 학습 전 모델은 제로샷과 같은 예측을 낸다.
- AdamW, 학습률 1e-5(인코더 마지막 블록) / 1e-3(분류층), 최대 10 epoch, 검증 손실이 2 epoch 연속 나아지지 않으면 종료.
- 증강은 무작위 자르기와 밝기 조정만 쓴다. 좌우 반전은 통행 방향과 글자를 뒤집으므로 쓰지 않는다.

In [ ]:
# 공통 셋업: 저장소 clone, 패키지 설치, Drive 연결
import os, sys
REPO_URL = "https://github.com/Cranezero/aix-assignment-pr.git"
REPO_DIR = "/content/aix-assignment-pr"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from src import config
from src.clip_utils import build_prompts, encode_texts, load_clip
from src.data import StreetDataset, get_transform, load_split, prepare_images
from src.metrics import evaluate, save_results
from src.utils import export_outputs, get_device, restore_outputs, set_seed

restore_outputs()  # results/, assets/ 폴더 생성 + Drive에 저장된 이전 결과 가져오기

set_seed(config.SEED)
device = get_device()
image_dir = prepare_images()
clip, processor = load_clip(device)
ckpt_dir = config.DRIVE_DIR / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

## 모델

`logits = scale * normalize(proj(vision(x))) @ W^T`. W를 문장 임베딩으로, scale을 CLIP의 logit scale로 두면 제로샷과 같은 식이 된다.

In [ ]:
class CLIPClassifier(nn.Module):
    def __init__(self, clip, text_emb, logit_scale):
        super().__init__()
        self.vision = clip.vision_model
        self.proj = clip.visual_projection
        self.head = nn.Linear(text_emb.shape[1], text_emb.shape[0], bias=False)
        self.head.weight.data.copy_(text_emb)
        self.register_buffer("scale", torch.tensor(float(logit_scale)))

    def forward(self, x):
        feat = self.proj(self.vision(pixel_values=x).pooler_output)
        return self.scale * self.head(F.normalize(feat, dim=-1))


text_emb = encode_texts(clip, processor, build_prompts("street"), device)
model = CLIPClassifier(clip, text_emb, clip.logit_scale.exp().item()).to(device)

# 마지막 블록과 분류층만 학습
for p in model.parameters():
    p.requires_grad = False
last_block = model.vision.encoder.layers[-1]
for p in list(last_block.parameters()) + list(model.head.parameters()):
    p.requires_grad = True

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터 {n_train:,} / 전체 {n_all:,} ({n_train / n_all:.1%})")

In [ ]:
splits = {s: load_split(s) for s in ["train", "val", "test"]}
g = torch.Generator().manual_seed(config.SEED)
train_loader = DataLoader(StreetDataset(splits["train"], image_dir, get_transform(train=True)),
                          batch_size=64, shuffle=True, num_workers=2, generator=g, drop_last=True)
eval_loader = {s: DataLoader(StreetDataset(splits[s], image_dir, get_transform(train=False)),
                             batch_size=256, shuffle=False, num_workers=2) for s in ["val", "test"]}


@torch.no_grad()
def predict(loader):
    model.eval()
    logits, labels = [], []
    for x, y in loader:
        with torch.autocast("cuda", dtype=torch.float16):
            logits.append(model(x.to(device)).float().cpu())
        labels.append(y)
    logits, labels = torch.cat(logits), torch.cat(labels)
    return logits, labels, F.cross_entropy(logits, labels).item()

## 학습 전 확인: 제로샷과 같은 지점에서 출발하는가

In [ ]:
logits0, test_labels, _ = predict(eval_loader["test"])
m0 = evaluate(logits0.softmax(-1).numpy(), test_labels.numpy())
zs = json.load(open(config.RESULTS_DIR / "zeroshot_metrics.json"))
print(f"학습 전 top1 {m0['top1']:.4f} / 제로샷 top1 {zs['top1']:.4f}")
assert abs(m0["top1"] - zs["top1"]) < 0.005, "초기화가 제로샷과 다르다"

## 학습

In [ ]:
MAX_EPOCHS, PATIENCE = 10, 2
optimizer = torch.optim.AdamW([
    {"params": last_block.parameters(), "lr": 1e-5},
    {"params": model.head.parameters(), "lr": 1e-3},
])
scaler = torch.amp.GradScaler("cuda")
trainable = {n for n, p in model.named_parameters() if p.requires_grad}

def save_ckpt():
    """학습하는 파라미터(마지막 블록, 분류층)만 저장한다."""
    torch.save({k: v.cpu() for k, v in model.state_dict().items() if k in trainable},
               ckpt_dir / "finetune_best.pt")

val_logits0, val_labels, val_loss0 = predict(eval_loader["val"])
val_acc0 = (val_logits0.argmax(1) == val_labels).float().mean().item()
history = [{"epoch": 0, "train_loss": np.nan, "val_loss": val_loss0, "val_acc": val_acc0, "sec": 0.0}]
best_loss, best_epoch, bad = val_loss0, 0, 0
save_ckpt()
t_start = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    t0, losses = time.time(), []
    for x, y in tqdm(train_loader, desc=f"epoch {epoch}", leave=False):
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = F.cross_entropy(model(x), y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
    val_logits, val_labels, val_loss = predict(eval_loader["val"])
    val_acc = (val_logits.argmax(1) == val_labels).float().mean().item()
    history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_loss": val_loss,
                    "val_acc": val_acc, "sec": time.time() - t0})
    print(history[-1])
    if val_loss < best_loss:
        best_loss, best_epoch, bad = val_loss, epoch, 0
        save_ckpt()
    else:
        bad += 1
        if bad >= PATIENCE:
            print("조기 종료")
            break

train_sec = time.time() - t_start
history = pd.DataFrame(history)
history.to_csv(config.RESULTS_DIR / "finetune_history.csv", index=False)
print(f"best epoch {best_epoch}, 학습 시간 {train_sec / 60:.1f}분")

## 학습 곡선

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
h = history[history["epoch"] > 0]
ax.plot(h["epoch"], h["train_loss"], "o-", label="train loss")
ax.plot(history["epoch"], history["val_loss"], "o-", label="val loss")
ax.axvline(best_epoch, color="gray", ls="--", lw=1, label=f"best (epoch {best_epoch})")
ax.set_xlabel("epoch"); ax.set_ylabel("cross-entropy loss"); ax.legend()
ax.set_title("Partial fine-tuning (epoch 0 = zero-shot init)")
plt.tight_layout()
plt.savefig(config.ASSETS_DIR / "learning_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 테스트 평가 (검증 손실이 가장 낮은 epoch의 가중치)

In [ ]:
missing, unexpected = model.load_state_dict(torch.load(ckpt_dir / "finetune_best.pt"), strict=False)
assert not unexpected and trainable.isdisjoint(missing)
logits, test_labels, _ = predict(eval_loader["test"])
metrics = save_results("finetune", splits["test"]["id"], logits.softmax(-1).numpy(), test_labels.numpy(),
                       extra={"best_epoch": best_epoch, "train_time_sec": train_sec,
                              "trainable_params": n_train})
print({k: round(v, 4) for k, v in metrics.items() if k in ("top1", "top5", "macro", "train_time_sec")})
per_country = pd.Series(metrics["per_country"]).rename("acc").to_frame()
per_country.insert(0, "name", [config.COUNTRIES[c] for c in per_country.index])
display(per_country.sort_values("acc", ascending=False).round(3))

In [ ]:
export_outputs()